<a href="https://colab.research.google.com/github/prodigal94/food-safe-bots/blob/main/EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
!pip install pyspark

In [28]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [21]:
# Initialize Spark with optimizations for large data
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("BDA_Project") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
file_path = '/content/drive/MyDrive/CSV/Trade_CropsLivestock_E_All_Data_(Normalized).csv'

df = spark.read.csv(file_path, header=True, inferSchema=True)

In [24]:
# Cache the dataframe to avoid recomputation
df.cache()

total_records = df.count()
print(f"Total records: {total_records:,}")
print(f"Total columns: {len(df.columns)}")

Total records: 17,270,631
Total columns: 14


In [33]:
print("\n--- Dataset Overview (BEFORE) ---")
# Check rows with missing values BEFORE
print("\n--- Rows with Missing Values (BEFORE) ---")

# Count rows that have ANY missing value
rows_with_missing = df.filter(
    F.greatest(*[F.when(F.col(c).isNull(), 1).otherwise(0) for c in df.columns]) == 1
).count()

rows_complete = total_records - rows_with_missing
print(f"  Complete rows (no missing): {rows_complete:,} ({rows_complete/total_records*100:.2f}%)")
print(f"  Rows with missing values: {rows_with_missing:,} ({rows_with_missing/total_records*100:.2f}%)")


--- Dataset Overview (BEFORE) ---

--- Rows with Missing Values (BEFORE) ---
  Complete rows (no missing): 952,262 (5.51%)
  Rows with missing values: 16,318,369 (94.49%)


In [30]:
# Get year range
min_year = df.select(F.min('Year')).collect()[0][0]
max_year = df.select(F.max('Year')).collect()[0][0]
print(f"Year range: {min_year} - {max_year}")

Year range: 1961 - 2024


In [31]:
# Check missing values BEFORE
print("\n--- Missing Values Count (BEFORE) ---")
missing_before = []
for col in df.columns:
    missing_count = df.filter(F.col(col).isNull()).count()
    missing_percent = (missing_count / total_records) * 100
    if missing_count > 0:
        print(f"  {col}: {missing_count:,} ({missing_percent:.2f}%)")
        missing_before.append((col, missing_count, missing_percent))


--- Missing Values Count (BEFORE) ---
  Note: 16,318,369 (94.49%)


In [32]:
# Check duplicates BEFORE
print("\n--- Duplicate Records (BEFORE) ---")
duplicates_before = total_records - df.distinct().count()
print(f"Duplicate rows: {duplicates_before:,}")


--- Duplicate Records (BEFORE) ---
Duplicate rows: 0
